# L04 Scipy - Exam Exercises
Yagmur Yesilyurt

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import integrate, optimize

## Q1: Simple numerical integral

In [ ]:
# the analytic answer for integral of exp(-x^2) from -inf to inf is sqrt(pi)
# over [-5, 5] it should be very close to that

f = lambda x: np.exp(-x**2)

result, error = integrate.quad(f, -5, 5)
print(f"quad result:  {result:.10f}")
print(f"sqrt(pi):     {np.sqrt(np.pi):.10f}")
print(f"error estimate: {error:.2e}")

In [ ]:
# now use Simpson's rule on sampled data and check how error shrinks with N
# Simpson is 4th order, so doubling N should reduce error by ~2^4 = 16

quad_ref = result  # use quad as ground truth

N_values = [8, 16, 32, 64, 128, 256, 512]
errors = []

for N in N_values:
    x = np.linspace(-5, 5, N)
    fvals = np.exp(-x**2)
    simp = integrate.simpson(fvals, x=x)
    errors.append(abs(simp - quad_ref))

# plot error vs N
fig, ax = plt.subplots(figsize=(6, 4))
ax.loglog(N_values, errors, "o-", label="Simpson error")

# reference line for 4th order convergence
N_arr = np.array(N_values, dtype=float)
ax.loglog(N_values, errors[0] * (N_values[0] / N_arr)**4, "--", label=r"$N^{-4}$ reference")

ax.set_xlabel("N (number of points)")
ax.set_ylabel("Error")
ax.set_title("Simpson's rule convergence")
ax.legend()
ax.grid(True, which="both", linestyle="--", alpha=0.4)
plt.tight_layout()
plt.savefig("L04_Q1_convergence.png", dpi=120)
plt.show()

The error follows the $N^{-4}$ line nicely — Simpson's rule is indeed 4th order accurate.

## Q3: Basins of attraction

In [ ]:
def q(x):
    return x**3 - 2*x**2 - 11*x + 12

# plot the function to get a rough idea where the roots are
x = np.linspace(-4, 5, 400)
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(x, q(x), label=r"$q(x) = x^3 - 2x^2 - 11x + 12$")
ax.axhline(0, color="k", linewidth=0.8)
ax.set_ylim(-30, 30)
ax.set_xlabel("x")
ax.set_ylabel("q(x)")
ax.legend()
ax.grid(True, linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()

In [ ]:
# from the plot: roots are somewhere near x=-3, x=1, x=4
# use brentq (needs a bracket where the function changes sign)

r1 = optimize.brentq(q, -4, -2)
r2 = optimize.brentq(q, 0, 2)
r3 = optimize.brentq(q, 3, 5)

print("Roots via brentq:")
print(f"  r1 = {r1:.6f}")
print(f"  r2 = {r2:.6f}")
print(f"  r3 = {r3:.6f}")

In [ ]:
# now find the same roots with optimize.root (Newton-like method)
# initial guess matters a lot here -- that's the whole point of basins of attraction

for x0 in [-3.5, 0.5, 4.0]:
    sol = optimize.root(q, x0)
    print(f"  x0={x0:5.1f}  ->  root = {sol.x[0]:.6f}  (converged: {sol.success})")

Both methods find the same three roots (~-3, ~1, ~4). `brentq` is safer because it needs a bracket, so it always converges to the root in that interval. `optimize.root` depends heavily on the initial guess — start near the wrong root and you land somewhere else.